In [2]:
import pandas as pd

DATA_PATH = r"C:\Users\fatem\OneDrive\Desktop\Dataset of Arabic Spam and Ham Tweets.csv"

df = pd.read_csv(DATA_PATH)
print(df.columns)
df.head()


Index(['Date', 'Time', 'Date Time', 'URL', 'Tweet Text', 'Cleaned Text',
       'User Name', 'Location', 'Replied Tweet ID ', 'Replied Tweet User ID',
       'Replied Tweet User name', 'Coordinates', 'Retweet Count',
       'Favorite Count', 'Favorited', 'Label'],
      dtype='object')


,Date,Time,Date Time,URL,Tweet Text,Cleaned Text,User Name,Location,Replied Tweet ID,Replied Tweet User ID,Replied Tweet User name,Coordinates,Retweet Count,Favorite Count,Favorited,Label
0,2021-03-02 00:00:00,06:48:15,Tue Mar 02 06:48:15 +0000 2021,https://twitter.com/AlArabiya/status/136664148...,سي إن إن تستعد إدارة الرئيس بايدن لفرض عقوبات ...,سي ان ان تستعد اداره الرئيس بايدن لفرض عقوبات ...,AlArabiya,NaN,NaN,NaN,NaN,NaN,4.0,20,False,Ham
1,2020-12-22 00:00:00,19:50:00,Mon Feb 22 19:50:00 +0000 2021,https://twitter.com/skynewsarabia/status/13639...,حكم يتصدى لكرة في طريقها لمرمى في لقطة كوميدية...,حكم يتصدي لكره في طريقها لمرمي في لقطه كوميديه...,skynewsarabia,"Abu Dhabi, UAE",NaN,NaN,NaN,NaN,5.0,36,False,Ham
2,2021-02-25 00:00:00,16:21:48,Thu Feb 25 16:21:48 +0000 2021,https://twitter.com/AlArabiya/status/136497388...,تابعونا على العربية عبر برنامج بانوراما ال بتو...,تابعونا علي العربيه عبر برنامج بانوراما ال بتو...,AlArabiya,NaN,NaN,NaN,NaN,NaN,4.0,19,False,Ham
3,2021-02-13 00:00:00,23:59:16,Sat Feb 13 23:59:16 +0000 2021,https://twitter.com/AlArabiya/status/136074035...,خبير بفريق التحقيق في منظمة الصحة العالمية بكي...,خبير بفريق التحقيق في منظمه الصحه العالميه بكي...,AlArabiya,NaN,NaN,NaN,NaN,NaN,7.0,38,False,Ham
4,2021-03-01 00:00:00,12:42:15,Mon Mar 01 12:42:15 +0000 2021,https://twitter.com/AlArabiya/status/136636818...,بالوثائق تعرف على أهم الاتفاقيات التاريخية لتر...,بالوثائق تعرف علي اهم الاتفاقيات التاريخيه لتر...,AlArabiya,NaN,NaN,NaN,NaN,NaN,4.0,16,False,Ham


In [4]:
df.columns = [str(c).strip() for c in df.columns]
print(df.columns.tolist())



['Date', 'Time', 'Date Time', 'URL', 'Tweet Text', 'Cleaned Text', 'User Name', 'Location', 'Replied Tweet ID', 'Replied Tweet User ID', 'Replied Tweet User name', 'Coordinates', 'Retweet Count', 'Favorite Count', 'Favorited', 'Label']


In [5]:
TEXT_COL  = "Cleaned Text"  
LABEL_COL = "Label"

df = df[[TEXT_COL, LABEL_COL]].dropna()
df[TEXT_COL] = df[TEXT_COL].astype(str)

print(df.head(3))
print(df[LABEL_COL].value_counts())


                                        Cleaned Text Label
0  سي ان ان تستعد اداره الرئيس بايدن لفرض عقوبات ...   Ham
1  حكم يتصدي لكره في طريقها لمرمي في لقطه كوميديه...   Ham
2  تابعونا علي العربيه عبر برنامج بانوراما ال بتو...   Ham
Label
Ham     11299
Spam     1940
Name: count, dtype: int64


In [6]:
def map_label(x):
    x = str(x).strip().lower()
    if x == "spam":
        return 1
    if x == "ham":
        return 0
    return None

df["label"] = df[LABEL_COL].apply(map_label)
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

df = df.rename(columns={TEXT_COL: "text"})
print(df["label"].value_counts())



label
0    11299
1     1940
Name: count, dtype: int64


In [7]:
print(df.shape)
print(df["label"].value_counts())
df.head()


(13239, 3)
label
0    11299
1     1940
Name: count, dtype: int64


,text,Label,label
0,سي ان ان تستعد اداره الرئيس بايدن لفرض عقوبات ...,Ham,0
1,حكم يتصدي لكره في طريقها لمرمي في لقطه كوميديه...,Ham,0
2,تابعونا علي العربيه عبر برنامج بانوراما ال بتو...,Ham,0
3,خبير بفريق التحقيق في منظمه الصحه العالميه بكي...,Ham,0
4,بالوثائق تعرف علي اهم الاتفاقيات التاريخيه لتر...,Ham,0


In [8]:
from sklearn.model_selection import train_test_split

tr_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

print("Train:", tr_df.shape)
print("Val  :", val_df.shape)
print(tr_df["label"].value_counts())


Train: (11915, 3)
Val  : (1324, 3)
label
0    10169
1     1746
Name: count, dtype: int64


In [9]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch

MODEL_NAME = "asafaya/bert-base-arabic"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ArabicSpamDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

C:\Users\fatem\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fatem\.cache\huggingface\hub\models--asafaya--bert-base-arabic. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [10]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_ds = ArabicSpamDataset(tr_df)
val_ds   = ArabicSpamDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Batches:", len(train_loader), len(val_loader))


Batches: 1490 166


In [11]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)

y_train = tr_df["label"].values
classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(weights, dtype=torch.float).to(device)
loss_fn = CrossEntropyLoss(weight=class_weights)

print("Class weights:", class_weights.cpu().numpy())


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/asafaya/bert-base-arabic/resolve/main/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /asafaya/bert-base-arabic/resolve/main/model.safetensors (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000254A6AA4CD0>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 00d84c30-f531-4635-ae9a-b6fe70514128)')' thrown while requesting GET https://huggingface.co/asafaya/bert-base-arabic/resolve/main/model.safetensors
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /asafaya/bert-base-arabic/resolve/main/model.safetensors (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000254A6FD41

model.safetensors:  54%|#####4    | 241M/445M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at asafaya/bert-base-arabic and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class weights: [0.58584917 3.4120848 ]


In [15]:
@torch.no_grad()
def predict_text(text):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc)
    return int(torch.argmax(out.logits, dim=1))

samples = [
    "مرحبا كيف حالك؟",
    "تهانينا! لقد ربحت جائزة مالية اضغط هنا",
    "تم تعليق حسابك البنكي، يرجى التحقق فوراً",
    "سأتصل بك لاحقاً",
    #spam
        "تهانينا! لقد ربحت جائزة مالية اضغط هنا لاستلامها",
    "تم تعليق حسابك البنكي، يرجى التحقق فوراً عبر هذا الرابط",
    "عاجل! فزت بهاتف آيفون جديد، اضغط هنا الآن",
    "تم اختيارك للحصول على قرض فوري بدون ضمانات",
    "لديك مبلغ مسترد بقيمة 500 دولار، أكد بياناتك الآن",
    "تحذير أمني: تم رصد نشاط مشبوه في حسابك",
    "اربح المال من المنزل بسهولة، سجل الآن",
    "تم إيقاف بطاقتك الائتمانية، يرجى التحقق فوراً",
    "اشترك الآن واحصل على خصم 90% لفترة محدودة",
    "رسالة مهمة: تم حظر حسابك مؤقتاً، اضغط لإعادة التفعيل",
    #ham
     "مرحبا كيف حالك؟",
    "سأتصل بك لاحقاً",
    "هل وصلت إلى المنزل بسلام؟",
    "لا تنسَ اجتماع الغد الساعة العاشرة",
    "شكراً جزيلاً على مساعدتك اليوم",
    "أنا في الطريق، سأصل بعد عشر دقائق",
    "عيد ميلاد سعيد، أتمنى لك يوماً رائعاً",
    "هل يمكنك إرسال الملف عندما تنتهي؟",
    "سأكون مشغولاً قليلاً، نكمل الحديث لاحقاً",
    "تم استلام رسالتك، شكراً"
]

for s in samples:
    print(s, "=>", predict_text(s), "(0=Ham, 1=Spam)")


مرحبا كيف حالك؟ => 0 (0=Ham, 1=Spam)
تهانينا! لقد ربحت جائزة مالية اضغط هنا => 0 (0=Ham, 1=Spam)
تم تعليق حسابك البنكي، يرجى التحقق فوراً => 1 (0=Ham, 1=Spam)
سأتصل بك لاحقاً => 0 (0=Ham, 1=Spam)
تهانينا! لقد ربحت جائزة مالية اضغط هنا لاستلامها => 0 (0=Ham, 1=Spam)
تم تعليق حسابك البنكي، يرجى التحقق فوراً عبر هذا الرابط => 1 (0=Ham, 1=Spam)
عاجل! فزت بهاتف آيفون جديد، اضغط هنا الآن => 0 (0=Ham, 1=Spam)
تم اختيارك للحصول على قرض فوري بدون ضمانات => 0 (0=Ham, 1=Spam)
لديك مبلغ مسترد بقيمة 500 دولار، أكد بياناتك الآن => 1 (0=Ham, 1=Spam)
تحذير أمني: تم رصد نشاط مشبوه في حسابك => 0 (0=Ham, 1=Spam)
اربح المال من المنزل بسهولة، سجل الآن => 0 (0=Ham, 1=Spam)
تم إيقاف بطاقتك الائتمانية، يرجى التحقق فوراً => 1 (0=Ham, 1=Spam)
اشترك الآن واحصل على خصم 90% لفترة محدودة => 0 (0=Ham, 1=Spam)
رسالة مهمة: تم حظر حسابك مؤقتاً، اضغط لإعادة التفعيل => 0 (0=Ham, 1=Spam)
مرحبا كيف حالك؟ => 0 (0=Ham, 1=Spam)
سأتصل بك لاحقاً => 0 (0=Ham, 1=Spam)
هل وصلت إلى المنزل بسلام؟ => 0 (0=Ham, 1=Spam)
لا تنسَ اجتماع 